# 03: CSR Interaction Matrix & Item-Item Collaborative Filtering
This notebook demonstrates constructing an implicit interaction matrix with event weights and damped logarithmic scaling, computing sparse cosine item similarity, and generating explainable recommendations.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

# Robust root resolver whether run from notebooks/ or workspace root
_curr = Path('.').resolve()
PROJECT_ROOT = _curr if (_curr / 'src').exists() else (_curr.parent if (_curr.parent / 'src').exists() else (_curr.parents[1] if len(_curr.parents) > 1 and (_curr.parents[1] / 'src').exists() else _curr))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Switch working directory to project root so all relative paths ('data/...', 'artifacts/...') resolve seamlessly
os.chdir(PROJECT_ROOT)

from src.features.interaction_features import InteractionMatrixBuilder
from src.recommenders.item_item import ItemItemRecommender


## 1. Build Damped Interaction Matrix

In [ ]:
train_df = pd.read_parquet('data/processed/train_events.parquet')
id_builder = InteractionMatrixBuilder().fit_mappings(train_df)
csr_mat, agg_df = id_builder.build_matrix(train_df)
print(f"Matrix shape: {csr_mat.shape}, Non-zeros: {csr_mat.nnz:,}")


## 2. Train Item-Item Collaborative Filtering

In [ ]:
item_model = ItemItemRecommender(k_nearest=50).fit(csr_mat, id_builder)
test_prod_id = list(id_builder.item_to_idx.keys())[0]
similar_items = item_model.get_similar_items(test_prod_id, n=5)
print(f"Top-5 items similar to Product #{test_prod_id}:")
for pid, sim in similar_items:
    print(f"  - Product #{pid}: Similarity = {sim:.4f}")


## 3. Recommend with Explainability Traces

In [ ]:
test_cust_id = list(id_builder.user_to_idx.keys())[0]
recs = item_model.recommend(test_cust_id, n=5)
print(f"Recommendations for Customer #{test_cust_id}:")
for pid, score in recs:
    explanation = item_model.explain(test_cust_id, pid)
    print(f"  - Product #{pid} (Score: {score:.3f}): {explanation['explanation']}")
